# A batch step that loses information

This notebook accompanies the docs page
[`lloyd-nonmonotone`](../../docs/examples/lloyd-nonmonotone.md). A terminal D partition is a
Mahalanobis Voronoi diagram in the metric its own retained information induces, so the obvious
algorithm is to iterate that assignment. The obvious algorithm is not monotone. The docs page
tells the story on eight rows; this notebook runs the full failure ledger, prints every table,
and re-renders the committed figure.

Set `SCOREQUANT_EXAMPLE_FAST=1` to shrink every sweep for a quick pass.

## Double precision is an application choice

The library never sets global numerical configuration at import time, so the notebook turns
double precision on itself, before anything computes.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

## The eight-row counterexample

Eight score rows, two parameters, three cells, equal weights, and a committed starting
labeling. Freezing the criterion metric and sending every row to its nearest cell mean moves
four rows — and lowers the criterion.

In [ ]:
import numpy as np

import scorequant as sq
from examples.lloyd_nonmonotone import (
    COUNTEREXAMPLE_BINS,
    COUNTEREXAMPLE_LABELS,
    COUNTEREXAMPLE_SCORES,
    counterexample_study,
    make_figure,
    run_study,
)

case = counterexample_study()

print(f"{'quantity':<34}{'before':>12}{'after':>12}{'change':>12}")
print("-" * 70)
print(
    f"{'log det of binned information':<34}{case.before:>12.6f}"
    f"{case.after:>12.6f}{case.step:>12.6f}"
)
print(
    f"{'frozen-metric distortion':<34}{case.distortion_before:>12.4f}"
    f"{case.distortion_after:>12.4f}{case.distortion_after - case.distortion_before:>12.4f}"
)
print()
print(f"rows relocated by the step        {case.moved}")
print(f"first-order surrogate change      {case.tangent_change:+.4f}")

The step improved the quantity a nearest-centroid assignment actually minimizes and made the
criterion worse. The reason is one line of convexity: the log determinant is concave, so its
first-order expansion is an upper bound, and improving an upper bound says nothing about the
function underneath it. The surrogate rose by more than eight while the criterion fell.

## What the guard does

The solver never accepts a proposal on the strength of the surrogate. It builds the proposal,
rebuilds the exact criterion state, and adopts it only if the exact objective strictly
improved.

In [ ]:
weights = np.full(COUNTEREXAMPLE_SCORES.shape[0], 1.0 / COUNTEREXAMPLE_SCORES.shape[0])
for guard in ("reject", "exchange"):
    result = sq.optimize_partition(
        COUNTEREXAMPLE_SCORES,
        weights=weights,
        n_bins=COUNTEREXAMPLE_BINS,
        config=sq.MahalanobisLloydConfig(seed=0, guard=guard),
        initial_labels=COUNTEREXAMPLE_LABELS,
    )
    history = np.asarray(result.objective_history) + case.whitening_offset
    built = f"{result.lloyd_iterations} / {result.accepted_lloyd_steps}"
    print(f"guard={guard!r}")
    print(f"  batch proposals built / accepted   {built}")
    print(f"  exchange scans / relocations       {result.scans} / {result.accepted_moves}")
    print(f"  exchange stable                    {result.exchange_stable}")
    print(f"  accepted trace (log det)           {[round(float(v), 6) for v in history]}")
    print()

Under `guard="reject"` one proposal is built, measured, and refused, and the solver reports
honestly that the labels it returns are not exchange-stable. Under the default
`guard="exchange"` the labels are handed to the exact positive-gain engine, which climbs
2.775392 nat in four single-row relocations, every one of them certified.

## The unguarded iteration, continued

The library stops after one rejected proposal. Running the batch step with no guard at all is
not something the library will do; the helper below exists only so the notebook can measure
what the guard is protecting against.

In [ ]:
from examples.lloyd_nonmonotone import unguarded_trajectory

run = unguarded_trajectory(
    COUNTEREXAMPLE_SCORES, weights, COUNTEREXAMPLE_LABELS, n_bins=COUNTEREXAMPLE_BINS
)
print(f"{'step':>6}{'log det':>14}{'change':>14}{'rows moved':>14}")
print("-" * 48)
print(f"{'start':>6}{run.objectives[0]:>14.6f}{'-':>14}{'-':>14}")
for index, value in enumerate(run.objectives[1:]):
    change = value - run.objectives[index]
    print(f"{index + 1:>6}{value:>14.6f}{change:>+14.6f}{run.moved[index]:>14}")
print()
print("stopped because the proposal was a", run.outcome, "point")
print("worst single step:", round(run.worst_step, 6))

On this fixture the unguarded iteration recovers: it dips, climbs, and lands on exactly the
labeling the guarded solver reaches. That is the weaker of the two things one might hope for.
Nothing made it recover, the dip is real and exactly measured, and nothing bounds the dip on a
larger table. The guarded path never goes down in the first place, which is a property rather
than an outcome.

## The whole study

`run_study` reruns the counterexample, sweeps the unguarded iteration from random starting
labels across three problems, three sample sizes, and two bin budgets, and runs one large
guarded fit from a random start. It is the same function that regenerates the committed JSON
and figure.

In [ ]:
study = run_study()
totals = study.metrics["ledger_totals"]

print(f"unguarded runs                       {totals['runs']}")
print(f"  ever stepped downhill              {totals['downhill_runs']}")
print(f"  vacated a cell                     {totals['emptied_runs']}")
print(f"  worst single-step change           {totals['worst_step']:+.6f}")

Two failure modes, two very different frequencies. The concavity failure that the eight-row
fixture exhibits is a small-sample, high-leverage effect and does not appear at all in this
sweep. Vacating a cell is common, and it is a state the exact criterion cannot represent: with
one cell empty the binned information is that of a smaller partition, singular as soon as the
cell count falls below the score dimension plus one. The same rule catches both — propose
freely, verify exactly, accept only improvements.

In [ ]:
print(f"{'problem':<26}{'events':>8}{'cells':>7}{'vacated a cell':>17}{'median steps':>14}")
print("-" * 72)
for row in study.metrics["ledger"]:
    vacated = f"{row['emptied_runs']} of {row['runs']}"
    print(
        f"{row['problem']:<26}{row['n_rows']:>8}{row['n_bins']:>7}"
        f"{vacated:>17}{row['median_steps']:>14.0f}"
    )

## What the guard costs at scale

Started from random labels on a large sample, the guarded batch has real work to do: it
crosses a bad initialization in a few dozen full-data relabelings and then hands over to
single-row exchange to settle the boundaries. Plain exchange reaches the same place, relocating
far more rows on the way.

In [ ]:
climb = study.metrics["climb"]
print(f"events                               {climb['n_rows']}")
print(f"cells                                {climb['n_bins']}")
print(f"start objective                      {climb['start_objective']:.6f}")
print()
print(f"{'solver':<22}{'full-data passes':>18}{'accepted steps':>17}{'objective':>13}")
print("-" * 70)
guarded_passes = f"{climb['lloyd_iterations']} + {climb['scans']}"
guarded_steps = f"{climb['accepted_lloyd_steps']} + {climb['accepted_moves']}"
print(
    f"{'guarded batch':<22}{guarded_passes:>18}{guarded_steps:>17}{climb['final_objective']:>13.6f}"
)
print(
    f"{'exact exchange':<22}{climb['exchange_scans']:>18}{climb['exchange_moves']:>17}"
    f"{climb['exchange_objective']:>13.6f}"
)
print()
print("every recorded step increased the objective:", climb["monotone"])

## The committed figure

In [ ]:
figure = make_figure(study)
figure

## Interpretation

Three things separate cleanly here.

Geometry that holds at an optimum is not a licence to iterate that geometry. The metric moves
with the partition, so a nearest-centroid step optimizes a surrogate that the criterion is
free to disagree with — and on eight rows it does, by 0.136521 nat.

The repair is not subtle and not optional. One exact rebuild per proposal turns a
non-monotone heuristic into a solver whose every recorded step is certified, and the same rule
also catches proposals that vacate a cell, which is the far more common failure at these
sample sizes.

The guarded batch is still worth having. From a bad initialization it makes large, coarse
moves that cross the whole configuration in a few full-data passes, and the exact exchange
then settles the boundary rows that a batch step keeps overshooting. The theory is
[Chapter 9](../../docs/book/ch09-mahalanobis-lloyd.md).